# Discritization

## Imports of required libs

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [3]:
df=pd.read_csv("Titanic-Dataset.csv",usecols=["Age","Fare","Survived"])
df.dropna(inplace=True)
df

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500
...,...,...,...
885,0,39.0,29.1250
886,0,27.0,13.0000
887,1,19.0,30.0000
889,1,26.0,30.0000


In [4]:
X=df.iloc[:,1:]
y=df.iloc[:,1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head(2)

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542


In [5]:
clf = DecisionTreeRegressor()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [6]:
accuracy_score(y_test, y_pred)

ValueError: continuous is not supported

In [7]:
np.mean(cross_val_score(DecisionTreeRegressor(), X, y, cv=10, scoring='accuracy'))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 140, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 388, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py", line 227, in accuracy_score
    y_type, y_true, y_pred = _check_targets(y

np.float64(nan)

## Using Equal Frequency (Quantile)

In [8]:
kbin_age = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')

In [9]:
trf = ColumnTransformer([
    ('first', kbin_age, [0]),
    ('second', kbin_fare, [1])
])


In [10]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)


In [15]:
trf.named_transformers_

{'first': KBinsDiscretizer(encode='ordinal', n_bins=10),
 'second': KBinsDiscretizer(encode='ordinal', n_bins=10)}

In [16]:
trf.named_transformers_['first'].n_bins
#10 bins has been created

10

In [18]:
trf.named_transformers_['second'].n_bins
#10 bins has been created

10

In [21]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.    ,   7.75  ,   7.8958,   9.225 ,  13.    ,  15.75  ,
               26.    ,  29.125 ,  51.4792,  82.1708, 512.3292])         ],
      dtype=object)

In [22]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42, 14.  , 19.  , 22.  , 25.  , 28.5 , 32.  , 36.  , 42.  ,
              50.  , 80.  ])                                                ],
      dtype=object)

In [31]:
output = pd.DataFrame({
    'age': X_train['Age'],
    'age_trf': X_train_trf[:, 0],
    'fare': X_train['Fare'],
    'fare_trf': X_train_trf[:, 1]
})
output['age_labels'] = pd.cut(x=X_train['Age'], 
                             bins=trf.named_transformers_['first'].bin_edges_[0].tolist())

output['fare_labels'] = pd.cut(x=X_train['Fare'], 
                             bins=trf.named_transformers_['second'].bin_edges_[0].tolist())


In [32]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
279,35.0,6.0,20.250,5.0,"(32.0, 36.0]","(15.75, 26.0]"
173,21.0,2.0,7.925,2.0,"(19.0, 22.0]","(7.896, 9.225]"
743,24.0,3.0,16.100,5.0,"(22.0, 25.0]","(15.75, 26.0]"
345,24.0,3.0,13.000,4.0,"(22.0, 25.0]","(9.225, 13.0]"
88,23.0,3.0,263.000,9.0,"(22.0, 25.0]","(82.171, 512.329]"


In [36]:
clf = DecisionTreeRegressor()
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test, y_pred2)

X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeRegressor, X, y, cv=10, scoring='accuracy'))

    kbin_age = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)
    kbin_fare = KBinsDiscretizer(n_bins=bins, encode='ordinal', strategy=strategy)


ValueError: continuous is not supported